# P105 - Product Cannibalization & Demand Shift Analysis

In [6]:
# IMPORTS AND DISPLAY SETUP

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


ModuleNotFoundError: No module named 'pandas'

In [ ]:
print('=' * 60)
print('  P105 | PHASE 2: DATA CLEANING & FEATURE ENGINEERING')
print('=' * 60)
# 1. LOAD RAW DATASET
df = pd.read_csv("D:\\Portfolio\\Creations\\Projects\\P105 Product Cannibalization & Demand Shift Analysis\\Data\\advanced_cannibalization_dataset.csv")

print('✅ Dataset loaded successfully.')
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")


  P105 | PHASE 2: DATA CLEANING & FEATURE ENGINEERING

✅ Dataset loaded successfully.
   Rows    : 60,000
   Columns : 13


In [ ]:
# 2. PART A - DATA CLEANING (TYPES, VALIDATIONS, SORTING)
print()
print('-' * 60)
print('  PART A | DATA CLEANING')
print('-' * 60)

# A1. Fix data types
df['Date'] = pd.to_datetime(df['Date'])
df['Stock_Available'] = df['Stock_Available'].astype(int)
df['Launch_Flag'] = df['Launch_Flag'].astype(int)
print('A1 complete: Date, Stock_Available, Launch_Flag types fixed.')

# A2. Validate Revenue = Price * Sales
df['Revenue_Check'] = (df['Price'] * df['Sales']).round(2)
mismatch = (df['Revenue'].round(2) != df['Revenue_Check']).sum()
print(f'A2 revenue mismatches: {mismatch}')
if mismatch > 0:
    df['Revenue'] = df['Revenue_Check']
df.drop(columns=['Revenue_Check'], inplace=True)

# A3. Validate rating range
invalid_ratings = df[(df['Rating'] < 1) | (df['Rating'] > 5)].shape[0]
print(f'A3 out-of-range ratings: {invalid_ratings}')

# A4. Validate positivity
neg_price = (df['Price'] <= 0).sum()
neg_sales = (df['Sales'] <= 0).sum()
neg_revenue = (df['Revenue'] <= 0).sum()
print(f'A4 Price <= 0: {neg_price} | Sales <= 0: {neg_sales} | Revenue <= 0: {neg_revenue}')

# A5. Standardize text columns
text_cols = ['Product_ID', 'Category', 'Product_Group', 'Region']
for col in text_cols:
    df[col] = df[col].str.strip()
print(f'A5 standardized text columns: {text_cols}')

# A6. Sort data
df.sort_values(['Product_ID', 'Date'], inplace=True)
df.reset_index(drop=True, inplace=True)
print('A6 complete: data sorted by Product_ID and Date.')

In [14]:
# 3. PART B - BASE FLAGS AND TIME FEATURES
print()
print('-' * 60)
print('  PART B | FEATURE ENGINEERING - BASE FLAGS')
print('-' * 60)

LAUNCH_DATE = pd.Timestamp('2024-06-01')
LAUNCHED_PRODUCTS = ['P1', 'P2', 'P3', 'P4', 'P5']
AFFECTED_GROUPS = ['G1', 'G2']

df['Period_Flag'] = np.where(df['Date'] < LAUNCH_DATE, 'Before_Launch', 'After_Launch')
df['Is_Launched_Product'] = df['Product_ID'].isin(LAUNCHED_PRODUCTS).astype(int)
df['Is_Affected_Group'] = df['Product_Group'].isin(AFFECTED_GROUPS).astype(int)

period_total_rev = df.groupby('Period_Flag')['Revenue'].transform('sum')
df['Revenue_Contribution_Pct'] = (df['Revenue'] / period_total_rev * 100).round(4)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Quarter'] = df['Date'].dt.quarter
df['Year_Month'] = df['Date'].dt.to_period('M').astype(str)

print(df[['Period_Flag', 'Is_Launched_Product', 'Is_Affected_Group', 'Year', 'Month', 'Quarter']].head().to_string(index=False))


------------------------------------------------------------
  PART B | FEATURE ENGINEERING - BASE FLAGS
------------------------------------------------------------


TypeError: '<' not supported between instances of 'str' and 'Timestamp'

In [ ]:
# 4. SALES CHANGE BEFORE VS AFTER (PRODUCT LEVEL)
sales_before = df[df['Period_Flag'] == 'Before_Launch'].groupby('Product_ID')['Sales'].mean().rename('Avg_Sales_Before')
sales_after = df[df['Period_Flag'] == 'After_Launch'].groupby('Product_ID')['Sales'].mean().rename('Avg_Sales_After')

sales_comparison = pd.concat([sales_before, sales_after], axis=1).round(2)
sales_comparison['Sales_Change_Pct'] = (
    (sales_comparison['Avg_Sales_After'] - sales_comparison['Avg_Sales_Before'])
    / sales_comparison['Avg_Sales_Before'] * 100
).round(2)

df = df.merge(sales_comparison[['Sales_Change_Pct']], on='Product_ID', how='left')
print(sales_comparison.sort_values('Sales_Change_Pct').to_string())

In [ ]:
# 5. CANNIBALIZATION FLAGS AND RATE
df['Cannibalization_Flag'] = (
    (df['Is_Launched_Product'] == 0)
    & (df['Is_Affected_Group'] == 1)
    & (df['Sales_Change_Pct'] < -5)
).astype(int)

cann_products = sorted(df[df['Cannibalization_Flag'] == 1]['Product_ID'].unique())

p6_before = df[(df['Product_ID'] == 'P6') & (df['Period_Flag'] == 'Before_Launch')]['Sales'].sum()
p6_after = df[(df['Product_ID'] == 'P6') & (df['Period_Flag'] == 'After_Launch')]['Sales'].sum()
p4p5_gained = df[(df['Product_ID'].isin(['P4', 'P5'])) & (df['Period_Flag'] == 'After_Launch')]['Sales'].sum()

months_before = df[df['Period_Flag'] == 'Before_Launch']['Year_Month'].nunique()
months_after = df[df['Period_Flag'] == 'After_Launch']['Year_Month'].nunique()
p6_loss_normalized = (p6_before / months_before) * months_after - p6_after
cann_rate = (p6_loss_normalized / p4p5_gained * 100) if p4p5_gained > 0 else 0

print(f'Cannibalized products: {cann_products}')
print(f'Cannibalized records: {df["Cannibalization_Flag"].sum():,}')
print(f'Cannibalization Rate: {cann_rate:.1f}%')

In [ ]:
# 6. EFFICIENCY, PRICE BAND, AND STOCK IMPACT
df['Marketing_Efficiency'] = np.where(
    df['Marketing_Spend'] > 0,
    (df['Revenue'] / df['Marketing_Spend']).round(4),
    np.nan
)

bins = [0, 50, 100, 200, 500]
labels = ['Budget (0-50)', 'Mid (51-100)', 'Premium (101-200)', 'Luxury (201+)']
df['Price_Band'] = pd.cut(df['Price'], bins=bins, labels=labels)

df['Stock_Impact_Flag'] = np.where(df['Stock_Available'] == 0, 1, 0)

print('Top 5 products by average Marketing_Efficiency:')
print(df.groupby('Product_ID')['Marketing_Efficiency'].mean().round(2).sort_values(ascending=False).head(5).to_string())
print()
print('Price_Band distribution:')
print(df['Price_Band'].value_counts().sort_index().to_string())
print()
print(f'Stock-out records: {df["Stock_Impact_Flag"].sum():,}')

In [ ]:
# 7. CUSTOMER SWITCHING PROXY AND DEMAND SHIFT FLAG
cust_p6_before = set(df[(df['Product_ID'] == 'P6') & (df['Period_Flag'] == 'Before_Launch')]['Customer_ID'])
cust_new_after = set(df[(df['Product_ID'].isin(['P4', 'P5'])) & (df['Period_Flag'] == 'After_Launch')]['Customer_ID'])
switchers = cust_p6_before.intersection(cust_new_after)

switching_rate = (len(switchers) / len(cust_p6_before) * 100) if cust_p6_before else 0

df['Demand_Shift_Flag'] = (
    df['Customer_ID'].isin(switchers)
    & df['Product_ID'].isin(['P4', 'P5'])
    & (df['Period_Flag'] == 'After_Launch')
).astype(int)

print(f'Customers who bought P6 before launch: {len(cust_p6_before):,}')
print(f'Customers who switched to P4/P5 after launch: {len(switchers):,}')
print(f'Customer Switching Rate: {switching_rate:.1f}%')
print(f'Demand shift records: {df["Demand_Shift_Flag"].sum():,}')

In [ ]:
# 8. SAVE CLEANED DATASET
output_path = Path('../Data/cannibalization_cleaned.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

new_cols = [
    'Period_Flag', 'Is_Launched_Product', 'Is_Affected_Group',
    'Revenue_Contribution_Pct', 'Year', 'Month', 'Month_Name',
    'Quarter', 'Year_Month', 'Sales_Change_Pct', 'Cannibalization_Flag',
    'Marketing_Efficiency', 'Price_Band', 'Stock_Impact_Flag', 'Demand_Shift_Flag'
]

print(f'Saved to: {output_path.resolve()}')
print(f'Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'New columns added: {len(new_cols)}')

In [ ]:
# 9. PHASE 2 SUMMARY CARD
print()
print('=' * 60)
print('  PHASE 2 SUMMARY - KEY METRICS DERIVED')
print('=' * 60)
print('Launch Date             : 2024-06-01')
print('Launched Products       : P1, P2, P3 (G1) | P4, P5 (G2)')
print(f'Cannibalized Products   : {sorted(df[df["Cannibalization_Flag"]==1]["Product_ID"].unique())}')
print(f'Cannibalization Rate    : {cann_rate:.1f}%')
print(f'Customer Switching Rate : {switching_rate:.1f}%')
print(f'Out-of-Stock Records    : {df["Stock_Impact_Flag"].sum():,}')
print(f'Total New Columns       : {15}')
print()
print('Phase 2 complete. Ready for Phase 3.')